# ORIENT'IA — Modèle d'orientation vers les filières de l'ISPM

Ce notebook construit un modèle qui, à partir du profil d'un lycéen/étudiant (série du bac, matière préférée, traits de personnalité RIASEC, centres d'intérêt), retourne les **3 à 5 filières de l'ISPM les plus adaptées avec un pourcentage de correspondance**.

## Approche (hybride)

1. **Similarité de contenu** : le profil de l'utilisateur est comparé (embeddings de texte multilingues) aux descriptions des 16 filières de l'ISPM (données embarquées dans ce notebook, extraites de `ispm-edu.md`).
2. **Classifieur RIASEC → département** : entraîné sur un grand dataset externe (Kaggle, ~145 000 réponses au test RIASEC de Holland), pour renforcer le lien traits de personnalité → domaine d'études.
3. **Filtre d'éligibilité** : pondération selon la série de bac de l'utilisateur, d'après les règles d'admission de l'ISPM.
4. **Calibration** : les poids du mélange sont ajustés en utilisant le sondage interne `Sondage.csv` (155 réponses) comme jeu de validation.

On n'entraîne **pas** de classifieur supervisé directement sur `Sondage.csv` : avec seulement 155 lignes pour 16 classes très déséquilibrées, un tel modèle mémoriserait le bruit plutôt que d'apprendre un vrai signal. Le sondage sert ici à *valider et calibrer* le pipeline, pas à l'entraîner.

## Avant de lancer sur Google Colab

1. Ouvrez ce notebook dans Colab (File → Upload notebook, ou glissez le fichier).
2. Dans le panneau de fichiers à gauche (icône dossier), **uploadez ces 2 fichiers** à la racine `/content/` :
   - `Sondage.csv` (fourni dans ce dossier)
   - `riasec.csv` — téléchargez le dataset Kaggle **[Holland Code (RIASEC) Test Responses](https://www.kaggle.com/datasets/lucasgreenwell/holland-code-riasec-test-responses)**, dézippez-le, et renommez le fichier de données principal (`data.csv` ou similaire) en `riasec.csv`.
3. Exécutez les cellules dans l'ordre (Runtime → Run all).

(Les 16 filières de l'ISPM et les règles d'admission sont déjà embarquées dans ce notebook — pas besoin d'uploader `ispm-edu.md`.)

La cellule suivante propose aussi un bouton d'upload direct si vous préférez ne pas glisser les fichiers manuellement.

In [1]:
!pip install -q sentence-transformers scikit-learn pandas numpy joblib

In [2]:
import re
import json
import joblib
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
# Chargement des fichiers. Sur Colab, si les fichiers ne sont pas déjà présents dans /content/,
# un bouton d'upload apparaît pour les deux fichiers requis.
REQUIRED_FILES = ["Sondage.csv", "riasec.csv"]

def ensure_files():
    import os
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
    if not missing:
        print("Tous les fichiers sont présents :", REQUIRED_FILES)
        return
    try:
        from google.colab import files
        print("Fichiers manquants :", missing)
        print("Uploadez-les maintenant (un ou plusieurs à la fois) :")
        uploaded = files.upload()
        still_missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
        if still_missing:
            print("Toujours manquants (vérifiez les noms exacts) :", still_missing)
    except ImportError:
        raise FileNotFoundError(
            f"Fichiers manquants : {missing}. Placez-les dans le même dossier que ce notebook."
        )

ensure_files()

sondage_df = pd.read_csv("Sondage.csv")
print("Sondage :", sondage_df.shape)

riasec_raw_df = pd.read_csv("riasec.csv", sep=None, engine="python")
print("RIASEC externe :", riasec_raw_df.shape)

Tous les fichiers sont présents : ['Sondage.csv', 'riasec.csv']
Sondage : (155, 11)


RIASEC externe : (145828, 94)


## 1. Les 16 filières de l'ISPM

Ces données (code, nom, département, domaine, description, mots-clés de chaque filière, et les séries de bac acceptées par département) viennent de `ispm-edu.md` mais sont embarquées ici en dur : ce sont des données de référence statiques, pas des données d'entraînement, donc pas besoin de re-uploader ce fichier à chaque run.

In [4]:
filieres = [
    {
        "code": "IGGLIA",
        "nom": "Informatique de Gestion, Génie Logiciel et Intelligence Artificielle",
        "departement": "Informatique et Télécommunications",
        "domaine": "Informatique, génie logiciel, systèmes d'information, intelligence artificielle.",
        "description": "Formation axée sur le développement logiciel, les systèmes d'information, l'informatique de gestion et l'intelligence artificielle. Elle prépare à la conception, au développement et à la maintenance d'applications et de solutions informatiques, ainsi qu'à l'utilisation des techniques d'intelligence artificielle.",
        "mots_cles": "informatique, gestion, génie logiciel, développement logiciel, systèmes d'information, intelligence artificielle, programmation, applications, IGGLIA."
    },
    {
        "code": "ESIIA",
        "nom": "Électronique, Systèmes Informatiques et Intelligence Artificielle",
        "departement": "Informatique et Télécommunications",
        "domaine": "Électronique, systèmes informatiques, systèmes embarqués, intelligence artificielle.",
        "description": "Combine l'électronique, les systèmes informatiques et l'intelligence artificielle. Porte sur la conception et l'exploitation de systèmes électroniques et informatiques intelligents, avec des applications dans l'automatisation, les systèmes embarqués et les technologies numériques.",
        "mots_cles": "électronique, informatique, systèmes embarqués, automatisation, systèmes intelligents, intelligence artificielle, technologies numériques, ESIIA."
    },
    {
        "code": "IMTICIA",
        "nom": "Informatique Multimédia, Technologies de l'Information et de la Communication et Intelligence Artificielle",
        "departement": "Informatique et Télécommunications",
        "domaine": "Informatique, multimédia, technologies de l'information et de la communication, intelligence artificielle.",
        "description": "Orientée vers l'informatique, les technologies de l'information et de la communication, le multimédia et l'intelligence artificielle. Permet de développer des compétences en technologies numériques, applications multimédias, communication digitale et systèmes intelligents.",
        "mots_cles": "informatique, multimédia, TIC, technologies numériques, communication digitale, intelligence artificielle, applications multimédias, IMTICIA."
    },
    {
        "code": "ISAIA",
        "nom": "Informatique, Statistique Appliquée et Intelligence Artificielle",
        "departement": "Informatique et Télécommunications",
        "domaine": "Informatique, statistiques, analyse de données, intelligence artificielle.",
        "description": "Combine informatique, statistiques appliquées et intelligence artificielle. Développe des compétences en analyse de données, modélisation statistique, programmation, apprentissage automatique et exploitation des données pour résoudre des problèmes complexes.",
        "mots_cles": "informatique, statistiques, statistique appliquée, analyse de données, data science, programmation, machine learning, apprentissage automatique, intelligence artificielle, ISAIA."
    },
    {
        "code": "EMII",
        "nom": "Électro-Mécanique et Informatique Industrielle",
        "departement": "Génie Industriel",
        "domaine": "Électrotechnique, mécanique, informatique industrielle, automatisation.",
        "description": "Se situe à l'intersection de l'électrotechnique, de la mécanique et de l'informatique industrielle. Prépare à la conception, au contrôle, à la maintenance et à l'automatisation des systèmes industriels.",
        "mots_cles": "électromécanique, électrotechnique, mécanique, informatique industrielle, automatisation, maintenance industrielle, systèmes industriels, EMII."
    },
    {
        "code": "ICMP",
        "nom": "Industries Chimiques, Minières et Pétrolières",
        "departement": "Génie Industriel",
        "domaine": "Industrie chimique, industrie minière, industrie pétrolière, procédés industriels.",
        "description": "Consacrée aux procédés industriels liés aux secteurs chimiques, miniers et pétroliers. Aborde les procédés de transformation, l'exploitation des ressources, le contrôle des processus et les aspects techniques de l'industrie extractive.",
        "mots_cles": "industrie chimique, industrie minière, industrie pétrolière, procédés industriels, transformation, exploitation minière, ressources naturelles, industrie extractive, ICMP."
    },
    {
        "code": "GCA",
        "nom": "Génie Civil et Architecture",
        "departement": "Génie Civil et Architecture",
        "domaine": "Génie civil, construction, architecture, infrastructures.",
        "description": "Dédiée à la conception, la construction et la gestion des ouvrages et infrastructures. Couvre les bâtiments, les structures, les infrastructures civiles, les matériaux et les aspects architecturaux des projets de construction.",
        "mots_cles": "génie civil, architecture, construction, bâtiments, structures, infrastructures, matériaux, ouvrages, GCA."
    },
    {
        "code": "CAA",
        "nom": "Commerce et Administration des Affaires",
        "departement": "Techniques des Affaires",
        "domaine": "Commerce, gestion, administration, management.",
        "description": "Orientée vers le commerce, la gestion et l'administration des organisations. Développe des compétences en management, marketing, gestion commerciale, administration et gestion des activités d'entreprise.",
        "mots_cles": "commerce, administration, gestion, management, marketing, gestion commerciale, entreprise, CAA."
    },
    {
        "code": "FIC",
        "nom": "Finances et Comptabilités",
        "departement": "Techniques des Affaires",
        "domaine": "Finance, comptabilité, gestion financière.",
        "description": "Spécialisée dans la gestion financière et comptable des organisations. Couvre la comptabilité, l'analyse financière, la gestion budgétaire, la fiscalité et le contrôle financier.",
        "mots_cles": "finance, comptabilité, gestion financière, analyse financière, budget, fiscalité, contrôle financier, FIC."
    },
    {
        "code": "DTJA",
        "nom": "Droit et Techniques Juridiques des Affaires",
        "departement": "Techniques des Affaires",
        "domaine": "Droit, droit des affaires, réglementation, juridique.",
        "description": "Axée sur l'application du droit aux activités économiques et commerciales. Développe des compétences en droit des affaires, réglementation, contrats, gestion des relations juridiques et traitement des questions juridiques des organisations.",
        "mots_cles": "droit, droit des affaires, droit commercial, réglementation, contrats, juridique, entreprises, DTJA."
    },
    {
        "code": "EMP",
        "nom": "Économie et Management de Projet",
        "departement": "Techniques des Affaires",
        "domaine": "Économie, gestion de projet, management.",
        "description": "Combine l'économie et la gestion de projets. Prépare à l'analyse économique, à la planification, au pilotage, au suivi et à l'évaluation de projets dans différents secteurs.",
        "mots_cles": "économie, management, gestion de projet, analyse économique, planification, pilotage, suivi de projet, évaluation, EMP."
    },
    {
        "code": "IAA",
        "nom": "Industrie Agroalimentaire",
        "departement": "Biotechnologie et Agronomie",
        "domaine": "Agroalimentaire, industrie alimentaire, transformation des produits agricoles.",
        "description": "Consacrée à la transformation, la conservation et la valorisation des produits agricoles et alimentaires. Aborde les procédés agroalimentaires, la qualité, la sécurité alimentaire et la gestion des productions industrielles.",
        "mots_cles": "agroalimentaire, industrie alimentaire, transformation, conservation, produits agricoles, qualité, sécurité alimentaire, production industrielle, IAA."
    },
    {
        "code": "AEE",
        "nom": "Agriculture et Élevage",
        "departement": "Biotechnologie et Agronomie",
        "domaine": "Agriculture, élevage, production agricole, gestion des exploitations.",
        "description": "Consacrée aux techniques de production agricole et d'élevage. Couvre la gestion des exploitations, les systèmes de production, l'amélioration des rendements et la gestion durable des ressources agricoles et animales.",
        "mots_cles": "agriculture, élevage, production agricole, exploitation agricole, rendement, ressources agricoles, ressources animales, développement durable, AEE."
    },
    {
        "code": "PIP",
        "nom": "Pharmacologie et Industries Pharmaceutiques",
        "departement": "Biotechnologie et Agronomie",
        "domaine": "Pharmacologie, industrie pharmaceutique, médicaments.",
        "description": "Orientée vers la pharmacologie et les procédés liés à l'industrie pharmaceutique. Porte sur les médicaments, les substances actives, les procédés de production, le contrôle qualité et les applications pharmaceutiques.",
        "mots_cles": "pharmacologie, pharmacie, industrie pharmaceutique, médicaments, substances actives, production pharmaceutique, contrôle qualité, PIP."
    },
    {
        "code": "TEE",
        "nom": "Tourisme et Environnement",
        "departement": "Tourisme",
        "domaine": "Tourisme, environnement, développement durable.",
        "description": "Combine le tourisme et la gestion de l'environnement. Prépare à la conception et à la gestion d'activités touristiques intégrant la préservation des ressources naturelles et le développement durable.",
        "mots_cles": "tourisme, environnement, écotourisme, développement durable, ressources naturelles, gestion touristique, TEE."
    },
    {
        "code": "TEH",
        "nom": "Tourisme et Hôtellerie",
        "departement": "Tourisme",
        "domaine": "Tourisme, hôtellerie, accueil, gestion touristique.",
        "description": "Consacrée aux métiers du tourisme et de l'hôtellerie. Développe des compétences en gestion hôtelière, accueil, organisation touristique, services, gestion des établissements et développement des activités touristiques.",
        "mots_cles": "tourisme, hôtellerie, accueil, gestion hôtelière, organisation touristique, services, établissements hôteliers, TEH."
    }
]

dept_series = {
    "Informatique et Télécommunications": "séries C, D, S et Techniques industrielles",
    "Génie Civil et Architecture": "séries C, D, S et Techniques du génie civil",
    "Biotechnologie et Agronomie": "séries C, D, S, Techniques agricoles, et série A2 avec note de mathématiques ≥ 12",
    "Techniques des Affaires": "toutes séries",
    "Génie Industriel": "séries C, D, S et Techniques industrielles",
    "Tourisme": "toutes séries"
}

print(f"{len(filieres)} filières chargées dans {len(dept_series)} départements.")
for f in filieres:
    print(f"  [{f['code']:>7}] {f['nom']}  —  {f['departement']}")

def eligibilite_serie(bac_serie, dept_name):
    """Score d'éligibilité souple (1.0 = série listée, 0.6 = partiellement incertain,
    0.4 = incertain/non listé). Ce n'est PAS une règle d'admission stricte, juste une
    pondération pour la recommandation."""
    txt = dept_series.get(dept_name, "").lower()
    if "toutes séries" in txt or "toutes series" in txt:
        return 1.0
    s = bac_serie.strip().upper()
    s_low = bac_serie.strip().lower()
    if s in ("C", "D", "S") and re.search(rf"\b{s}\b", dept_series.get(dept_name, "")):
        return 1.0
    if s == "TGI" and "technique" in txt and "industr" in txt:
        return 1.0
    # séries techniques nommées en toutes lettres (ex: "Techniques agricoles",
    # "Techniques du génie civil") — non reconnues par le seul code "TGI" ci-dessus.
    if "technique" in s_low and "agricol" in s_low and "agricol" in txt:
        return 1.0
    if "technique" in s_low and "civil" in s_low and "civil" in txt:
        return 1.0
    # Série A2 : seule cette sous-série précise (avec note de maths ≥ 12, non modélisée
    # ici faute de donnée) est admise. "A" seule ou les autres sous-séries (A1, A3, A4)
    # restent incertaines plutôt que de recevoir le plein score par défaut.
    if s == "A2" and "a2" in txt:
        return 1.0
    if s.startswith("A") and "a2" in txt:
        return 0.6
    return 0.4

# petit test
for s in ["C", "D", "A", "A2", "TGI", "Techniques agricoles", "Techniques du génie civil", "L"]:
    print(s, "->", eligibilite_serie(s, "Informatique et Télécommunications"))
print("Techniques agricoles ->", eligibilite_serie("Techniques agricoles", "Biotechnologie et Agronomie"))
print("Techniques du génie civil ->", eligibilite_serie("Techniques du génie civil", "Génie Civil et Architecture"))

16 filières chargées dans 6 départements.
  [ IGGLIA] Informatique de Gestion, Génie Logiciel et Intelligence Artificielle  —  Informatique et Télécommunications
  [  ESIIA] Électronique, Systèmes Informatiques et Intelligence Artificielle  —  Informatique et Télécommunications
  [IMTICIA] Informatique Multimédia, Technologies de l'Information et de la Communication et Intelligence Artificielle  —  Informatique et Télécommunications
  [  ISAIA] Informatique, Statistique Appliquée et Intelligence Artificielle  —  Informatique et Télécommunications
  [   EMII] Électro-Mécanique et Informatique Industrielle  —  Génie Industriel
  [   ICMP] Industries Chimiques, Minières et Pétrolières  —  Génie Industriel
  [    GCA] Génie Civil et Architecture  —  Génie Civil et Architecture
  [    CAA] Commerce et Administration des Affaires  —  Techniques des Affaires
  [    FIC] Finances et Comptabilités  —  Techniques des Affaires
  [   DTJA] Droit et Techniques Juridiques des Affaires  —  Techniques

## 2. Nettoyage du sondage ISPM (`Sondage.csv`)

- Le champ *Caractère(s)* est une question à choix multiples au format `"Catégorie : item1, item2, ..., Catégorie2 : item1, ..."`. On en extrait simplement les lettres RIASEC sélectionnées (R/I/A/S/E/C).
- Le champ *parcours suivi / domaine d'exercice* est lui aussi à choix multiples, en texte libre, et ne correspond pas exactement aux 16 codes de filières. On le mappe vers les **6 départements** de l'ISPM (plus fiable qu'un mapping filière-exact vu l'ambiguïté des réponses comme « Informatique / TIC » qui recouvre 4 filières différentes). Les réponses hors périmètre ISPM (ex: « Santé », « aéronautique ») sont ignorées pour la calibration.

In [5]:
RIASEC_MAP = {
    "réaliste": "R", "realiste": "R",
    "investigateur": "I",
    "artistique": "A",
    "social": "S",
    "entreprenant": "E",
    "conventionnel": "C",
}
riasec_cat_regex = re.compile("|".join(RIASEC_MAP.keys()), re.IGNORECASE)

def parse_riasec_letters(text):
    if not isinstance(text, str):
        return set()
    return {RIASEC_MAP[m.group(0).lower()] for m in riasec_cat_regex.finditer(text)}

DOMAIN_TO_DEPT = {
    "informatique / technologies de l'information et de la communication": "Informatique et Télécommunications",
    "intelligence artificielle": "Informatique et Télécommunications",
    "électronique / systèmes informatiques": "Informatique et Télécommunications",
    "économie et management de projet": "Techniques des Affaires",
    "industrie agroalimentaire": "Biotechnologie et Agronomie",
    "génie civil et architecture": "Génie Civil et Architecture",
    "finances et comptabilité": "Techniques des Affaires",
    "génie industriel / électro-mécanique industrielle": "Génie Industriel",
    "droit et techniques juridiques des affaires": "Techniques des Affaires",
    "industries chimiques, minières et pétrolières": "Génie Industriel",
    "agriculture et élevage": "Biotechnologie et Agronomie",
    "commerce et administration des affaires": "Techniques des Affaires",
    "pharmacologie et industries pharmaceutiques": "Biotechnologie et Agronomie",
    "hôtellerie": "Tourisme",
    "tourisme": "Tourisme",
    "environnement": "Tourisme",
}

def _norm_apostrophes(s):
    return s.replace("\u2019", "'")  # apostrophe typographique -> apostrophe simple

DOMAIN_TO_DEPT = {_norm_apostrophes(k): v for k, v in DOMAIN_TO_DEPT.items()}

def map_target_to_departments(raw_text):
    if not isinstance(raw_text, str):
        return set()
    text = _norm_apostrophes(raw_text.lower())
    return {dept for label, dept in DOMAIN_TO_DEPT.items() if label in text}

COL_SERIE = "Quelle était votre série en Terminale ?"
COL_MATIERE = "Quelle a été votre matière préférée au Lycée?"
COL_TARGET = [c for c in sondage_df.columns if c.startswith("Si vous êtes étudiant")][0]
COL_CARACTERE = "Quel(s) Caractère(s) définissent votre personnalité ?"
COL_INTERETS = [c for c in sondage_df.columns if "centres d" in c][0]

clean = sondage_df[[COL_SERIE, COL_MATIERE, COL_TARGET, COL_CARACTERE, COL_INTERETS]].copy()
clean.columns = ["serie", "matiere", "cible_brute", "caractere_brut", "interets"]
clean = clean.dropna(subset=["serie", "matiere", "cible_brute", "caractere_brut"])

clean["riasec_letters"] = clean["caractere_brut"].apply(parse_riasec_letters)
clean["departements_cibles"] = clean["cible_brute"].apply(map_target_to_departments)

n_total = len(clean)
n_labelled = (clean["departements_cibles"].apply(len) > 0).sum()
print(f"{n_total} lignes nettoyées, dont {n_labelled} avec au moins un département ISPM reconnu "
      f"(utilisables pour la calibration).")
clean.head()

155 lignes nettoyées, dont 149 avec au moins un département ISPM reconnu (utilisables pour la calibration).


,serie,matiere,cible_brute,caractere_brut,interets,riasec_letters,departements_cibles
0,C,Mathématique et Physique-Chimie,Education,"Entreprenant : Diriger, Convaincre, Negocier",Football- Enseignement- Lecture.,{E},{}
1,C,"Mathématiques, SVT",Informatique / Technologies de l’Information e...,"Investigateur : Comprendre, Analyser, Resoudre...",Domaine de la tech,"{A, I, C}",{Informatique et Télécommunications}
2,D,Mathematique,Finances et Comptabilité,"Entreprenant : Diriger, Convaincre, Negocier",Develeppement personnel,{E},{Techniques des Affaires}
3,C,"Maths, PC",Électronique / Systèmes Informatiques,"Realiste : Manipuler, Construire, Reparer, Art...",Argents,"{A, R, E}",{Informatique et Télécommunications}
4,TGI,Électrotechnique,Génie Industriel / Électro-Mécanique Industrielle,"Investigateur : Comprendre, Analyser, Resoudre",Randonnée,{I},{Génie Industriel}


## 3. Classifieur RIASEC → département (données externes Kaggle)

Le dataset [Holland Code (RIASEC) Test Responses](https://www.kaggle.com/datasets/lucasgreenwell/holland-code-riasec-test-responses) contient ~145 000 réponses à 48 items (8 par catégorie R/I/A/S/E/C, échelle 1–5) ainsi qu'une majeure d'études déclarée en texte libre. On :

1. calcule le score moyen par catégorie RIASEC pour chaque répondant ;
2. regroupe la majeure déclarée en une des 6 grandes familles correspondant aux départements de l'ISPM (par mots-clés) ;
3. entraîne un `RandomForestClassifier` (6 scores RIASEC → département) sur les lignes dont la majeure a pu être classée.

Ce classifieur capture un vrai signal généralisable (dizaines de milliers d'exemples), contrairement à un modèle entraîné sur les 155 lignes du sondage ISPM.

In [6]:
# Détection automatique des colonnes R1..R8, I1..I8, A1..A8, S1..S8, E1..E8, C1..C8
riasec_cols = {}
for letter in "RIASEC":
    cols = [c for c in riasec_raw_df.columns if re.fullmatch(rf"{letter}\d+", str(c).strip())]
    riasec_cols[letter] = sorted(cols, key=lambda c: int(re.search(r"\d+", c).group()))

print({k: len(v) for k, v in riasec_cols.items()})
assert all(len(v) > 0 for v in riasec_cols.values()), (
    "Colonnes RIASEC introuvables — vérifiez les noms de colonnes de riasec.csv avec riasec_raw_df.columns"
)

riasec_scores = pd.DataFrame({
    letter: riasec_raw_df[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    for letter, cols in riasec_cols.items()
})

major_col_candidates = [c for c in riasec_raw_df.columns if str(c).strip().lower() == "major"]
major_col = major_col_candidates[0] if major_col_candidates else None
assert major_col is not None, "Colonne 'major' introuvable dans riasec.csv"

MAJOR_KEYWORDS_TO_DEPT = {
    "Informatique et Télécommunications": [
        "computer", "software", "information technology", r"\bit\b", "informatics",
        "telecommunicat", "electrical engineering", "electronics", "programming",
        "data science", "artificial intelligence", r"\bai\b", "computing",
    ],
    "Génie Industriel": [
        "mechanical engineering", "industrial engineering", "manufacturing",
        "chemical engineering", "mining engineering", "petroleum",
    ],
    "Génie Civil et Architecture": [
        "civil engineering", "architecture", "construction management", "structural engineering",
    ],
    "Techniques des Affaires": [
        "business", "commerce", "management", "finance", "accounting", "economics",
        r"\blaw\b", "marketing", "administration",
    ],
    "Biotechnologie et Agronomie": [
        "biology", "agricultur", "agronomy", "food science", "biotechnology",
        "pharmacy", "pharmacolog", "veterinary", "animal science",
    ],
    "Tourisme": [
        "tourism", "hospitality", "hotel management",
    ],
}
compiled_keywords = {
    dept: re.compile("|".join(kws), re.IGNORECASE) for dept, kws in MAJOR_KEYWORDS_TO_DEPT.items()
}

def major_to_dept(major_text):
    if not isinstance(major_text, str) or not major_text.strip():
        return None
    matches = [dept for dept, rgx in compiled_keywords.items() if rgx.search(major_text)]
    return matches[0] if len(matches) == 1 else None  # on ignore les majeures ambiguës (plusieurs matches)

riasec_scores["departement"] = riasec_raw_df[major_col].apply(major_to_dept)
labelled = riasec_scores.dropna(subset=["departement"]).copy()

print(f"{len(labelled)} / {len(riasec_scores)} lignes RIASEC externes avec un département reconnu.")
print(labelled["departement"].value_counts())

# Départements avec trop peu d'exemples pour un split stratifié fiable (voire pour que
# train_test_split ne plante pas) : on les exclut de l'entraînement plutôt que de laisser
# le RandomForest apprendre du bruit sur quelques lignes.
MIN_SAMPLES_PER_CLASS = 20
class_counts = labelled["departement"].value_counts()
rare_depts = class_counts[class_counts < MIN_SAMPLES_PER_CLASS].index.tolist()
if rare_depts:
    print(f"\n⚠️  Départements avec moins de {MIN_SAMPLES_PER_CLASS} exemples externes, "
          f"exclus de l'entraînement du classifieur RIASEC : {rare_depts}")
    print("   Ces départements n'auront jamais de score RIASEC (seul le score de contenu s'appliquera).")
    labelled = labelled[~labelled["departement"].isin(rare_depts)].copy()

{'R': 8, 'I': 8, 'A': 8, 'S': 8, 'E': 8, 'C': 8}


26123 / 145828 lignes RIASEC externes avec un département reconnu.
departement
Techniques des Affaires               15305
Informatique et Télécommunications     3794
Biotechnologie et Agronomie            3502
Génie Civil et Architecture            1760
Génie Industriel                       1420
Tourisme                                342
Name: count, dtype: int64


In [7]:
X = labelled[list("RIASEC")].values
y = labelled["departement"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

riasec_clf = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
riasec_clf.fit(X_train, y_train)

y_pred = riasec_clf.predict(X_test)
print("Accuracy test :", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

DEPT_ORDER = list(riasec_clf.classes_)

missing_depts = set(dept_series.keys()) - set(DEPT_ORDER)
if missing_depts:
    print(f"⚠️  Départements ISPM absents du classifieur RIASEC : {missing_depts}\n"
          "   Leurs filières n'auront jamais de score RIASEC (uniquement le score de contenu).")

Accuracy test : 0.5222966507177034
                                    precision    recall  f1-score   support

       Biotechnologie et Agronomie       0.41      0.62      0.49       701
       Génie Civil et Architecture       0.14      0.15      0.15       352
                  Génie Industriel       0.11      0.10      0.11       284
Informatique et Télécommunications       0.27      0.32      0.29       759
           Techniques des Affaires       0.76      0.64      0.70      3061
                          Tourisme       0.00      0.00      0.00        68

                          accuracy                           0.52      5225
                         macro avg       0.28      0.31      0.29      5225
                      weighted avg       0.55      0.52      0.53      5225



## 4. Embeddings de contenu des 16 filières

On encode le texte de chaque filière (domaine + description + mots-clés) avec un modèle d'embeddings de phrases multilingue, capable de bien traiter le français.

In [8]:
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

for f in filieres:
    f["texte"] = f"{f['domaine']}. {f['description']} Mots-clés : {f['mots_cles']}"

filiere_embeddings = embed_model.encode(
    [f["texte"] for f in filieres], normalize_embeddings=True, show_progress_bar=True
)
print("Matrice d'embeddings des filières :", filiere_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Matrice d'embeddings des filières : (16, 384)


## 5. Pipeline de prédiction

`predict_filieres(...)` combine :

- la similarité cosinus entre le profil texte de l'utilisateur et chaque filière (normalisée 0–1) ;
- la probabilité de département donnée par le classifieur RIASEC, redistribuée entre les filières du département au prorata de leur similarité de contenu ;
- un multiplicateur d'éligibilité selon la série de bac ;

puis normalise le tout en pourcentages (somme = 100 %) et retourne le top-k.

In [9]:
FILIERE_DEPTS = np.array([f["departement"] for f in filieres])

def riasec_letters_to_vector(letters, selected_score=4.0, baseline=2.0):
    """Convertit un sous-ensemble de lettres RIASEC sélectionnées en vecteur 6-dim
    sur la même échelle (1-5) que les scores moyens du dataset externe."""
    return np.array([selected_score if l in letters else baseline for l in "RIASEC"])

def normalize_0_1(x):
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-9:
        return np.ones_like(x) / len(x)
    return (x - lo) / (hi - lo)

def _score_components(serie, matiere, interets, caractere_letters):
    """Calcule séparément les 3 composantes du score (similarité de contenu, similarité
    RIASEC redistribuée par filière, éligibilité) sans les combiner. Séparé de
    predict_filieres() pour que la calibration (section 6) puisse réutiliser les embeddings
    déjà calculés au lieu de ré-encoder le texte du profil à chaque poids testé."""
    # 1) similarité de contenu
    profil_txt = (
        f"J'aime {matiere}. Mes centres d'intérêt : {interets}. "
        f"Traits de personnalité : {', '.join(sorted(caractere_letters)) or 'non précisé'}."
    )
    profil_emb = embed_model.encode([profil_txt], normalize_embeddings=True)
    sim_content = cosine_similarity(profil_emb, filiere_embeddings)[0]
    sim_content_norm = normalize_0_1(sim_content)

    # 2) probabilité RIASEC -> département, redistribuée sur les filières du département
    riasec_vec = riasec_letters_to_vector(caractere_letters).reshape(1, -1)
    dept_proba = dict(zip(DEPT_ORDER, riasec_clf.predict_proba(riasec_vec)[0]))

    sim_riasec = np.zeros(len(filieres))
    for dept in set(FILIERE_DEPTS):
        idx = np.where(FILIERE_DEPTS == dept)[0]
        p_dept = dept_proba.get(dept, 0.0)
        share = normalize_0_1(sim_content[idx])
        share = share / share.sum() if share.sum() > 0 else np.ones(len(idx)) / len(idx)
        sim_riasec[idx] = p_dept * share

    # 3) éligibilité selon la série de bac
    elig = np.array([eligibilite_serie(serie, f["departement"]) for f in filieres])

    return sim_content_norm, sim_riasec, elig


def predict_filieres(serie, matiere, interets, caractere_letters,
                      w_content=0.5, w_riasec=0.5, top_k=5, verbose=False):
    sim_content_norm, sim_riasec, elig = _score_components(serie, matiere, interets, caractere_letters)

    score = (w_content * sim_content_norm + w_riasec * sim_riasec) * elig
    proba = score / score.sum() if score.sum() > 0 else np.ones(len(filieres)) / len(filieres)

    if verbose:
        for f, s, r, e in zip(filieres, sim_content_norm, sim_riasec, elig):
            print(f"{f['code']:>7} content={s:.3f} riasec={r:.3f} elig={e:.2f}")

    ranking = sorted(zip(filieres, proba), key=lambda t: t[1], reverse=True)[:top_k]
    return [
        {"code": f["code"], "nom": f["nom"], "departement": f["departement"], "pourcentage": round(100 * p, 1)}
        for f, p in ranking
    ]

## 6. Calibration sur le sondage ISPM

On utilise les lignes du sondage dont la réponse a pu être rattachée à au moins un département ISPM (voir section 2) comme jeu de validation. Métrique : **taux de réussite top-3** = la vraie réponse (département) apparaît-elle parmi les départements des 3 filières les mieux classées ?

Avec seulement ~100 lignes de validation, ce résultat est indicatif, pas une preuve statistique solide — à ré-évaluer quand le sondage aura plus de réponses.

In [10]:
eval_rows = clean[clean["departements_cibles"].apply(len) > 0]
print(f"{len(eval_rows)} lignes utilisées pour la calibration.")

# Composantes pré-calculées une seule fois par ligne (indépendantes de w_content) pour éviter
# de ré-encoder le texte du profil 11 fois (une par poids testé) : ~11x plus rapide.
eval_components = [
    (_score_components(row["serie"], row["matiere"], row["interets"], row["riasec_letters"]),
     row["departements_cibles"])
    for _, row in eval_rows.iterrows()
]

def top3_hit_rate(w_content):
    w_riasec = 1 - w_content
    hits = 0
    for (sim_content_norm, sim_riasec, elig), target_depts in eval_components:
        score = (w_content * sim_content_norm + w_riasec * sim_riasec) * elig
        top3_idx = np.argsort(score)[::-1][:3]
        pred_depts = {FILIERE_DEPTS[i] for i in top3_idx}
        if pred_depts & target_depts:
            hits += 1
    return hits / len(eval_components)

results = {}
for w in np.arange(0.0, 1.01, 0.1):
    results[round(w, 1)] = top3_hit_rate(w)
    print(f"w_content={w:.1f}  ->  top-3 hit rate = {results[round(w, 1)]:.2%}")

BEST_W_CONTENT = max(results, key=results.get)
BEST_W_RIASEC = round(1 - BEST_W_CONTENT, 1)
print(f"\nMeilleur poids retenu : w_content={BEST_W_CONTENT}, w_riasec={BEST_W_RIASEC} "
      f"(top-3 hit rate = {results[BEST_W_CONTENT]:.2%})")

149 lignes utilisées pour la calibration.


w_content=0.0  ->  top-3 hit rate = 58.39%
w_content=0.1  ->  top-3 hit rate = 79.19%
w_content=0.2  ->  top-3 hit rate = 77.85%
w_content=0.3  ->  top-3 hit rate = 79.87%
w_content=0.4  ->  top-3 hit rate = 80.54%
w_content=0.5  ->  top-3 hit rate = 79.87%
w_content=0.6  ->  top-3 hit rate = 79.19%
w_content=0.7  ->  top-3 hit rate = 77.85%
w_content=0.8  ->  top-3 hit rate = 77.85%
w_content=0.9  ->  top-3 hit rate = 77.85%
w_content=1.0  ->  top-3 hit rate = 76.51%

Meilleur poids retenu : w_content=0.4, w_riasec=0.6 (top-3 hit rate = 80.54%)


## 7. Démo

Exemples de profils, avec les poids calibrés à l'étape précédente.

In [11]:
exemples = [
    dict(serie="D", matiere="Mathématiques", interets="jeux vidéo, robotique, programmation",
         caractere_letters={"I", "R"}),
    dict(serie="A", matiere="Économie", interets="entrepreneuriat, marketing, réseaux sociaux",
         caractere_letters={"E", "C"}),
    dict(serie="D", matiere="SVT", interets="agriculture, nature, environnement",
         caractere_letters={"R", "S"}),
]

for ex in exemples:
    print("Profil :", ex)
    preds = predict_filieres(**ex, w_content=BEST_W_CONTENT, w_riasec=BEST_W_RIASEC, top_k=5)
    for p in preds:
        print(f"   {p['pourcentage']:>5.1f}%  {p['code']:>7}  {p['nom']}  ({p['departement']})")
    print()

Profil : {'serie': 'D', 'matiere': 'Mathématiques', 'interets': 'jeux vidéo, robotique, programmation', 'caractere_letters': {'I', 'R'}}
    14.0%    ISAIA  Informatique, Statistique Appliquée et Intelligence Artificielle  (Informatique et Télécommunications)
    12.3%  IMTICIA  Informatique Multimédia, Technologies de l'Information et de la Communication et Intelligence Artificielle  (Informatique et Télécommunications)
    11.9%   IGGLIA  Informatique de Gestion, Génie Logiciel et Intelligence Artificielle  (Informatique et Télécommunications)
    10.0%    ESIIA  Électronique, Systèmes Informatiques et Intelligence Artificielle  (Informatique et Télécommunications)
     9.9%     EMII  Électro-Mécanique et Informatique Industrielle  (Génie Industriel)

Profil : {'serie': 'A', 'matiere': 'Économie', 'interets': 'entrepreneuriat, marketing, réseaux sociaux', 'caractere_letters': {'E', 'C'}}


    17.7%      CAA  Commerce et Administration des Affaires  (Techniques des Affaires)
    15.4%      EMP  Économie et Management de Projet  (Techniques des Affaires)
    13.3%      FIC  Finances et Comptabilités  (Techniques des Affaires)
     9.1%      TEE  Tourisme et Environnement  (Tourisme)
     8.2%      TEH  Tourisme et Hôtellerie  (Tourisme)

Profil : {'serie': 'D', 'matiere': 'SVT', 'interets': 'agriculture, nature, environnement', 'caractere_letters': {'R', 'S'}}


    11.8%      GCA  Génie Civil et Architecture  (Génie Civil et Architecture)
     9.7%      IAA  Industrie Agroalimentaire  (Biotechnologie et Agronomie)
     8.7%      AEE  Agriculture et Élevage  (Biotechnologie et Agronomie)
     7.9%      TEH  Tourisme et Hôtellerie  (Tourisme)
     7.7%  IMTICIA  Informatique Multimédia, Technologies de l'Information et de la Communication et Intelligence Artificielle  (Informatique et Télécommunications)



In [12]:
# Cellule interactive (facultative) — à exécuter dans Colab pour tester votre propre profil.
def demo_interactive():
    print("Série au bac (ex: C, D, S, A, TGI, L) :")
    serie = input("> ").strip()
    print("Matière préférée au lycée :")
    matiere = input("> ").strip()
    print("Centres d'intérêt (texte libre) :")
    interets = input("> ").strip()
    print("Traits de personnalité parmi Réaliste(R) Investigateur(I) Artistique(A) Social(S) Entreprenant(E) Conventionnel(C),")
    print("séparés par des virgules (ex: I,R) :")
    lettres = {t.strip().upper() for t in input("> ").split(",") if t.strip()}

    preds = predict_filieres(serie, matiere, interets, lettres,
                              w_content=BEST_W_CONTENT, w_riasec=BEST_W_RIASEC, top_k=5)
    print("\nFilières recommandées :")
    for p in preds:
        print(f"   {p['pourcentage']:>5.1f}%  {p['code']:>7}  {p['nom']}  ({p['departement']})")

# Décommentez la ligne suivante pour lancer le mode interactif :
# demo_interactive()

## 8. Sauvegarde du pipeline

Sauvegarde du classifieur RIASEC, des embeddings de filières et des poids calibrés, pour pouvoir réutiliser le modèle sans tout ré-entraîner (ex: dans une petite API ou une appli).

In [13]:
joblib.dump(
    {
        "riasec_clf": riasec_clf,
        "dept_order": DEPT_ORDER,
        "filieres": filieres,
        "filiere_embeddings": filiere_embeddings,
        "dept_series": dept_series,
        "w_content": BEST_W_CONTENT,
        "w_riasec": BEST_W_RIASEC,
    },
    "orientia_pipeline.joblib",
)
print("Pipeline sauvegardé dans orientia_pipeline.joblib")
print("(le modèle SentenceTransformer se re-télécharge automatiquement au rechargement, il n'est pas inclus dans le fichier)")

Pipeline sauvegardé dans orientia_pipeline.joblib
(le modèle SentenceTransformer se re-télécharge automatiquement au rechargement, il n'est pas inclus dans le fichier)
